<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Import-Standard-Data-Science-Packages" data-toc-modified-id="Import-Standard-Data-Science-Packages-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Import Standard Data Science Packages</a></span><ul class="toc-item"><li><span><a href="#Set-Colorblind-Friendly-Palette" data-toc-modified-id="Set-Colorblind-Friendly-Palette-1.1"><span class="toc-item-num">1.1&nbsp;&nbsp;</span>Set Colorblind-Friendly Palette</a></span></li></ul></li><li><span><a href="#Load-and-Visualise-dPCR-Data" data-toc-modified-id="Load-and-Visualise-dPCR-Data-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>Load and Visualise dPCR Data</a></span></li><li><span><a href="#Data-Analysis" data-toc-modified-id="Data-Analysis-3"><span class="toc-item-num">3&nbsp;&nbsp;</span>Data Analysis</a></span><ul class="toc-item"><li><span><a href="#Organise-Data" data-toc-modified-id="Organise-Data-3.1"><span class="toc-item-num">3.1&nbsp;&nbsp;</span>Organise Data</a></span></li><li><span><a href="#Define-ACA-Model-(Neural-Network)" data-toc-modified-id="Define-ACA-Model-(Neural-Network)-3.2"><span class="toc-item-num">3.2&nbsp;&nbsp;</span>Define ACA Model (Neural Network)</a></span></li><li><span><a href="#Train-and-Compute-Performance-(10-Fold-Cross-Val)" data-toc-modified-id="Train-and-Compute-Performance-(10-Fold-Cross-Val)-3.3"><span class="toc-item-num">3.3&nbsp;&nbsp;</span>Train and Compute Performance (10-Fold Cross Val)</a></span></li><li><span><a href="#Visualise-AMCA-Coefficients" data-toc-modified-id="Visualise-AMCA-Coefficients-3.4"><span class="toc-item-num">3.4&nbsp;&nbsp;</span>Visualise AMCA Coefficients</a></span></li><li><span><a href="#Compute-Performance-as-a-Function-of-Training-Volume" data-toc-modified-id="Compute-Performance-as-a-Function-of-Training-Volume-3.5"><span class="toc-item-num">3.5&nbsp;&nbsp;</span>Compute Performance as a Function of Training Volume</a></span></li></ul></li></ul></div>

# Import Standard Data Science Packages

In [8]:
%load_ext autoreload
%autoreload
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from pathlib import Path

import chip_utilities as utils

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Set Colorblind-Friendly Palette

In [9]:
import matplotlib as mpl
from cycler import cycler
from matplotlib.colors import to_hex
mpl.rcParams['axes.prop_cycle'] = cycler(color=[to_hex(i) for i in
                                                [(0, 0.45, 0.70),    # Blue
                                                (0.9, 0.6, 0.0),     # Orange
                                                (0.0, 0.60, 0.50),    # Bluish Green
                                                (0.8, 0.4, 0),       # Vermillion/Burnt Orange
                                                (0.35, 0.7, 0.9),    # Sky Blue
                                                (0.8, 0.6, 0.7),     # Reddish Purple/Pink
                                                (0, 0, 0),           # Black
                                                (0.5, 0.5, 0.5),     # Grey
                                                (0.286, 0, 0.573),   # Deep Purple
                                                (0.8, 0.1, 0.3)]])   # 10th: Berry Red

# Load and Visualise dPCR Data

In [10]:
exp_folder = "/Users/kautsarg/Library/CloudStorage/OneDrive-ImperialCollegeLondon/Final Project/Run Data/trial test data/"
exp_path = Path(exp_folder, "D20250808_E00_C00_F4500KHz_U_Sample_7")
sigmoid_data_path = f'{exp_path}/normalised_ori_ac_df.csv'

chip_data = pd.read_csv(sigmoid_data_path)
chip_data = utils.denormalized(chip_data)
chip_data.head()

,row_min,row_max,row_range,well,Cycle71.10000000000001,Cycle74.89999999999999,Cycle78.7,Cycle82.49999999999999,Cycle86.3,Cycle90.10000000000001,...,Cycle2364.4,Cycle2368.2000000000003,Cycle2372.0,Cycle2375.8,Cycle2379.6000000000004,Cycle2383.4,Cycle2387.1000000000004,Cycle2390.9,Cycle2394.7000000000003,Cycle2398.5
0,0.209451,0.239304,0.029854,0,0.219883,0.225581,0.221032,0.223199,0.226792,0.220374,...,0.237564,0.233608,0.236609,0.236990,0.234911,0.234537,0.236800,0.237373,0.238334,0.233423
1,0.197048,0.227095,0.030047,0,0.219883,0.225896,0.222948,0.219505,0.218379,0.214700,...,0.223727,0.223143,0.222368,0.219316,0.215609,0.224511,0.221597,0.225499,0.223923,0.226494
2,0.190823,0.223456,0.032633,0,0.219883,0.215283,0.218522,0.220078,0.218136,0.211950,...,0.221062,0.219298,0.223255,0.216987,0.221062,0.220864,0.221259,0.222253,0.221259,0.223456
3,0.201863,0.221085,0.019222,0,0.219883,0.218566,0.215485,0.221085,0.219090,0.216118,...,0.214233,0.215359,0.215233,0.216373,0.212877,0.213738,0.213245,0.216245,0.214607,0.210822
4,0.203428,0.228837,0.025409,0,0.219883,0.223504,0.219256,0.222212,0.222212,0.211433,...,0.227704,0.225473,0.226359,0.224812,0.225473,0.224593,0.228837,0.226137,0.225473,0.225915


# Data Analysis
## Organise Data

In [11]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegressionCV

X_AC = chip_data.filter(like="Cycle").values

encoder = LabelEncoder()
encoder.fit(chip_data['well'])
ytrue = encoder.transform(chip_data['well'])

y_lim_min, y_lim_max = chip_data.filter(like="Cycle").values.min(), chip_data.filter(like="Cycle").values.max()
X_AC.shape, ytrue.shape

((12047, 615), (12047,))

## Define ACA Model (Neural Network)

In [12]:
import tensorflow as tf
from scikeras.wrappers import KerasClassifier
import math
class myWrapper(KerasClassifier):
    # def predict(self, X):
    #     return self.model.predict(X).argmax(axis=1)
    
    # def predict_proba(self, X):
    #     return self.model.predict(X)
    pass

display(f'TF VERSION: {tf.__version__}')

def create_model(kernel_size_1=5): 
    
    # TIMESTEPS, CHANNELS
    inputs = tf.keras.layers.Input(shape=(X_AC.shape[1], 1))
    
    # Inject the parameters into the kernel_size arguments
    x = tf.keras.layers.Conv1D(32, kernel_size_1, activation='relu')(inputs)
    x = tf.keras.layers.Conv1D(16, math.ceil(kernel_size_1/2), activation='relu')(x)
    
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(len(np.unique(ytrue)), activation='softmax')(x)
    
    model = tf.keras.models.Model(inputs=inputs, outputs=x)
    model.compile(optimizer='adam', 
                  loss='sparse_categorical_crossentropy', 
                  metrics=['accuracy'])
    return model

'TF VERSION: 2.21.0'

In [13]:
# def deeper_CNN_model(input_shape, num_classes, kernel_size_1=5): 
    
#     inputs = tf.keras.layers.Input(shape=input_shape)
    
#     # --- BLOCK 1 ---
#     # padding='same' ensures we don't arbitrarily shrink the sequence length before pooling
#     x = tf.keras.layers.Conv1D(64, kernel_size_1, padding='same')(inputs)
#     x = tf.keras.layers.BatchNormalization()(x)
#     x = tf.keras.layers.Activation('relu')(x)
#     x = tf.keras.layers.MaxPooling1D(pool_size=2)(x)
#     x = tf.keras.layers.SpatialDropout1D(0.2)(x)
    
#     # --- BLOCK 2 ---
#     # Deeper layer, increased filters, smaller kernel
#     x = tf.keras.layers.Conv1D(128, math.ceil(kernel_size_1/2), padding='same')(x)
#     x = tf.keras.layers.BatchNormalization()(x)
#     x = tf.keras.layers.Activation('relu')(x)
#     x = tf.keras.layers.MaxPooling1D(pool_size=2)(x)
#     x = tf.keras.layers.SpatialDropout1D(0.2)(x)

#     # --- BLOCK 3 ---
#     # Final feature extraction block
#     x = tf.keras.layers.Conv1D(128, math.ceil(kernel_size_1/2), padding='same')(x)
#     x = tf.keras.layers.BatchNormalization()(x)
#     x = tf.keras.layers.Activation('relu')(x)
    
#     # --- CLASSIFICATION HEAD ---
#     # Replaces Flatten() to drastically reduce parameters and prevent overfitting
#     x = tf.keras.layers.GlobalAveragePooling1D()(x)
    
#     # Intermediate dense layer with standard dropout
#     x = tf.keras.layers.Dense(64, activation='relu')(x)
#     x = tf.keras.layers.Dropout(0.3)(x)
    
#     # Output layer
#     outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    
#     model = tf.keras.models.Model(inputs=inputs, outputs=outputs)
#     model.compile(optimizer='adam', 
#                   loss='sparse_categorical_crossentropy', 
#                   metrics=['accuracy'])
    
#     return model

## Train and Compute Performance (10-Fold Cross Val)

In [14]:
# from sklearn.model_selection import GridSearchCV, StratifiedKFold
# import math

# N_SPLITS = 10
# skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=0)

# param_grid = {
#     'model__kernel_size_1': [5, 10, 25, 50, 75],
#     'batch_size': [512],
# }

# clf_AC = myWrapper(
#     model=create_model, 
#     epochs=1000, 
#     batch_size=512,
#     shuffle=True,
#     verbose=False
# )

# grid = GridSearchCV(
#     estimator=clf_AC, 
#     param_grid=param_grid, 
#     cv=skf,
#     scoring='accuracy',
#     verbose=9,
#     # n_jobs=-1
# )

# grid_result = grid.fit(X_AC, ytrue)

In [15]:
# import pandas as pd
# import numpy as np
# from pathlib import Path
# from sklearn.model_selection import GridSearchCV, StratifiedKFold
# import math
# import joblib
# import json

# N_SPLITS = 10
# skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=0)

# param_grid = {
#     'model__kernel_size_1': [5, 10, 15, 20],
#     'batch_size': [512, 256],
# }

# input_shape = (X_AC.shape[1], 1)
# class_num = len(np.unique(ytrue))

# clf_dCNN = myWrapper(
#     model=deeper_CNN_model, 
#     model__input_shape=input_shape,
#     model__num_classes=class_num,
#     epochs=1000, 
#     batch_size=512,
#     shuffle=True,
#     verbose=False
# )

# grid_dCNN = GridSearchCV(
#     estimator=clf_dCNN, 
#     param_grid=param_grid, 
#     cv=skf,
#     scoring='accuracy',
#     verbose=9,
#     # n_jobs=-1
# )

# grid_result_dCNN = grid_dCNN.fit(X_AC, ytrue)

# exp_folder = "/Users/kautsarg/Library/CloudStorage/OneDrive-ImperialCollegeLondon/Final Project/Run Data/trial test data/"
# exp_path = Path(exp_folder) / "D20250808_E00_C00_F4500KHz_U_Sample_7"

# exp_path.mkdir(parents=True, exist_ok=True)

# metrics_path = exp_path / 'dcnn_cross_val.csv'
# json_path = exp_path / 'dcnn_cross_val.json'
# model_path = exp_path / 'dcnn_cross_val.joblib'

# results_df = pd.DataFrame(grid_dCNN.cv_results_)
# results_df.to_csv(metrics_path, index=False)

# best_meta = {
#     'best_params': grid_dCNN.best_params_,
#     'best_score': float(grid_dCNN.best_score_),
#     'n_splits': N_SPLITS
# }

# with open(json_path, 'w') as f:
#     json.dump(best_meta, f, indent=4)

# joblib.dump(grid_dCNN.best_estimator_, model_path)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import GridSearchCV, StratifiedShuffleSplit
import math
import joblib
import json
import tensorflow as tf
import gc

def deeper_CNN_model(input_shape, num_classes, kernel_size_1=5): 
    # CRITICAL: Do not comment these out during GridSearchCV
    tf.keras.backend.clear_session() 
    gc.collect()
    
    inputs = tf.keras.layers.Input(shape=input_shape)
    
    # --- BLOCK 1 ---
    x = tf.keras.layers.Conv1D(32, kernel_size_1, padding='same')(inputs)
    x = tf.keras.layers.Activation('relu')(x)
    
    # --- BLOCK 2 ---
    x = tf.keras.layers.Conv1D(64, math.ceil(kernel_size_1/2), padding='same')(x)
    x = tf.keras.layers.Activation('relu')(x)

    # --- BLOCK 3 ---
    x = tf.keras.layers.Conv1D(128, math.ceil(kernel_size_1/2), padding='same')(x)
    
    # --- CLASSIFICATION HEAD ---
    # CRITICAL: Use GlobalAveragePooling1D to prevent parameter explosion
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    
    # Keep Dense layer units reasonable. 128 is plenty.
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    
    model = tf.keras.models.Model(inputs=inputs, outputs=outputs)
    
    model.compile(optimizer=tf.keras.optimizers.Adam(), 
                  loss='sparse_categorical_crossentropy', 
                  metrics=['accuracy'])
    
    return model

# CRITICAL: Uncomment this. float64 takes 2x the memory. Apple Silicon needs float32.
X_AC = X_AC.astype(np.float32)
ytrue = ytrue.astype(np.float32)

cv_split = StratifiedShuffleSplit(n_splits=1, test_size=0.10, random_state=0)

param_grid = {
    'model__kernel_size_1': [5, 10, 15, 20, 25],
    'batch_size': [512], 
}

input_shape = (X_AC.shape[1], 1)
class_num = len(np.unique(ytrue))

# Highly recommended to uncomment EarlyStopping. 1000 epochs is likely 900 more than you need.
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='loss', 
    patience=15, 
    restore_best_weights=True,
    verbose=1
)

clf_dCNN = myWrapper(
    model=deeper_CNN_model, 
    model__input_shape=input_shape,
    model__num_classes=class_num,
    epochs=1000, 
    shuffle=True,
    verbose=False,
    # callbacks=[early_stopping] # Re-added
)

grid_dCNN = GridSearchCV(
    estimator=clf_dCNN, 
    param_grid=param_grid, 
    cv=cv_split, 
    scoring='accuracy',
    verbose=9,
)

X_AC_3D = X_AC.reshape((X_AC.shape[0], X_AC.shape[1], 1))

grid_result_dCNN = grid_dCNN.fit(X_AC_3D, ytrue)

exp_folder = "/Users/kautsarg/Library/CloudStorage/OneDrive-ImperialCollegeLondon/Final Project/Run Data/trial test data/"
exp_path = Path(exp_folder) / "D20250808_E00_C00_F4500KHz_U_Sample_7"

exp_path.mkdir(parents=True, exist_ok=True)

metrics_path = exp_path / 'dcnn_train_test_simplified.csv'
json_path = exp_path / 'dcnn_train_test_simplified.json'
model_path = exp_path / 'dcnn_train_test_simplified.joblib'

results_df = pd.DataFrame(grid_dCNN.cv_results_)
results_df.to_csv(metrics_path, index=False)

best_meta = {
    'best_params': grid_dCNN.best_params_,
    'best_score': float(grid_dCNN.best_score_),
    'split_type': '90-10 Train-Test' 
}

with open(json_path, 'w') as f:
    json.dump(best_meta, f, indent=4)

joblib.dump(grid_dCNN.best_estimator_, model_path)

Fitting 1 folds for each of 5 candidates, totalling 5 fits
[CV 1/1] END batch_size=512, model__kernel_size_1=5;, score=0.736 total time=60.4min


In [ ]:
import pandas as pd
import json

exp_folder = "/Users/kautsarg/Library/CloudStorage/OneDrive-ImperialCollegeLondon/Final Project/Run Data/trial test data/"
exp_path = Path(exp_folder, "D20250808_E00_C00_F4500KHz_U_Sample_7")
crossval_data_path = f'{exp_path}/cnn_cross_val.csv'

# 1. Save the full CV results (mean scores, std dev, fold results)
results_df = pd.DataFrame(grid.cv_results_)
results_df.to_csv(crossval_data_path, index=False)

# 2. Save the best parameters as a JSON (easy for humans to read)
best_meta = {
    'best_params': grid.best_params_,
    'best_score': float(grid.best_score_),
    'n_splits': N_SPLITS
}

crossval_data_path = f'{exp_path}/cnn_cross_val.json'
with open(crossval_data_path, 'w') as f:
    json.dump(best_meta, f, indent=4)

In [ ]:
# import joblib
# import pandas as pd

# # Load the metrics for plotting/analysis
# df = pd.read_csv('full_grid_search_metrics.csv')

# # Load the model for inference or further testing
# model = joblib.load('trained_best_model.joblib')

# # Verify it works
# # model.predict(X_test)

In [ ]:
import joblib

exp_folder = "/Users/kautsarg/Library/CloudStorage/OneDrive-ImperialCollegeLondon/Final Project/Run Data/trial test data/"
exp_path = Path(exp_folder, "D20250808_E00_C00_F4500KHz_U_Sample_7")
crossval_data_path = f'{exp_path}/cnn_cross_val.joblib'

joblib.dump(grid.best_estimator_, crossval_data_path)

In [ ]:
from sklearn.metrics import accuracy_score
import statsmodels.api as sm
from scipy.stats import bartlett, ttest_ind

acc_for_each_fold = lambda x: [accuracy_score(i, j) for i, j in zip(x, y_trues_)]

acc_dict = {'FFI': acc_for_each_fold(y_preds_FFI_),
            'ACA': acc_for_each_fold(y_preds_AC_),
            # 'MCA': acc_for_each_fold(y_preds_MC_),
            # 'AMCA': acc_for_each_fold(y_preds_)
            }
acc = pd.DataFrame(acc_dict)

normality_test = lambda x: sm.stats.lilliefors(x)[1]

# variance_test = lambda x: bartlett(x, acc['AMCA'])[1]

# ttest = lambda x: ttest_ind(x, acc['AMCA'], equal_var=False)[1]

# statistics = lambda x: [np.mean(x), np.std(x), normality_test(x), variance_test(x), ttest(x)]

# stats = pd.DataFrame({k: statistics(acc[k]) for k in acc.columns})

# stats.index = ['Mean Acc.', 'Std Acc.', 'Normality p-val.', 'Equal Var. p-val', 'Ttest p-val']

# pd.set_option('display.float_format', lambda x: f'{x:.4f}')

# stats

In [ ]:
classes = encoder.classes_

ypreds_methods = [y_preds_FFI_, y_preds_AC_, y_preds_AC_kNN_] #, y_preds_MC_, y_preds_]
titles = ['Final Fluorescence Intensity',
          'Amplification Curve Analysis',
          'Amplification Curve Analysis (kNN)'] #,
        #   'Melting Curve Analysis',
        #   'Amplification and Melting Curve Analysis']
short_names = ['FFI', 'ACA_ConvNN', 'ACA_kNN']#, 'MCA', 'AMCA']
accs_mean = []
accs_std = []
for yps in ypreds_methods:
    acc_temp = []
    for yp, y_tr in zip(yps, y_trues_):
        acc_temp.append(accuracy_score(y_tr, yp)*100)
    accs_mean.append(np.mean(acc_temp))
    accs_std.append(np.std(acc_temp))

custom_labels = [f'{a:.2f} ± {s:.2f}' for a, s in zip(accs_mean, accs_std)]

fig, ax = plt.subplots(figsize=(8, 6))
bars = ax.bar(short_names, accs_mean, yerr=accs_std, capsize=6, edgecolor='black')
ax.bar_label(bars, labels=custom_labels, padding=0, fontsize=10)
ax.set_ylim(0, max(accs_mean) + max(accs_std) + 3)
ax.set_ylabel('Accuracy')
ax.set_title('Model Performance')

plt.show()

In [ ]:
print(short_names)
print(accs_mean)
print(accs_std)

In [ ]:
classes = encoder.classes_

ypreds_methods = [y_preds_FFI, y_preds_AC, y_preds_AC_kNN] #, y_preds_MC, y_preds]
titles = ['Final Fluorescence Intensity',
          'Amplification Curve Analysis',
          'Amplification Curve Analysis (kNN)']
        #   ,
        #   'Melting Curve Analysis',
        #   'Amplification and Melting Curve Analysis']
short_names = ['FFI', 'ACA_ConvNN', 'ACA_kNN'] #, 'MCA', 'AMCA']

for yp, title, short_name in zip(ypreds_methods, titles, short_names):

    fig, ax = plt.subplots(1, 1, figsize=(5, 5), dpi=300)

    utils.plot_confusion_matrix(y_trues, yp, classes, ax, normalize=False)
    ax.set_title(f'{title}\n'+ax.get_title(), fontsize=14, weight='bold')
    ax.set_ylabel('True Target', fontsize=14, weight='bold')
    ax.set_xlabel('Predicted Target', fontsize=14, weight='bold')

    plt.tight_layout()
    plt.savefig(f'Images/dPCR_{short_name}.pdf')
    plt.show()

In [ ]:
clf_AC_all = myWrapper(build_fn=create_model, 
                            epochs=1000, 
                            batch_size=512, 
                            shuffle=True,
                            verbose=False)
clf_AC_all.fit(X_AC, ytrue)
y_pred = clf_AC_all.predict(X_AC) 


In [ ]:
from sklearn.metrics import accuracy_score
print(accuracy_score(ytrue, y_pred))

In [ ]:
trained_keras_model = clf_AC_all.model_ 


conv1_layer = trained_keras_model.layers[1] 

# get_weights() returns a list: [weights_array, biases_array]
weights, biases = conv1_layer.get_weights()

print(f"Layer 1 Weights shape: {weights.shape}") # Should be (5, 1, 16)

# 3. Plot the 16 filters in a 4x4 grid
n_filters = weights.shape[2]

fig, axes = plt.subplots(4, 4, figsize=(12, 10))
fig.suptitle('Visualization of 1st Conv1D Layer Kernels (Size: 5)', fontsize=16, weight='bold')

for i, ax in enumerate(axes.flatten()):
    if i < n_filters:
        # Extract the kernel: all 5 timesteps, channel 0, filter i
        kernel_values = weights[:, 0, i] 
        
        # Plot as a line with markers
        ax.plot(kernel_values, marker='o', color='b', linewidth=2)
        ax.set_title(f'Filter {i}')
        ax.set_xticks(range(weights.shape[0]))
        ax.axhline(0, color='black', linewidth=0.5, linestyle='--') # Add a zero line
    
    ax.grid(True, alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
conv2_layer = trained_keras_model.layers[2]
weights_2, _ = conv2_layer.get_weights()
# Shape is (3, 16, 8)

fig, axes = plt.subplots(2, 4, figsize=(15, 6))
fig.suptitle('2nd Conv1D Layer Kernels as Heatmaps', fontsize=16, weight='bold')

for i, ax in enumerate(axes.flatten()):
    # Extract the 2D matrix for filter i: shape (3, 16)
    kernel_matrix = weights_2[:, :, i] 
    
    cax = ax.imshow(kernel_matrix, aspect='auto', cmap='viridis')
    ax.set_title(f'Filter {i}')
    ax.set_xlabel('Input Channels (from L1)')
    ax.set_ylabel('Kernel Steps')

plt.tight_layout()
plt.show()

In [ ]:
true_label

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

# 1. Access the trained Keras model
trained_keras_model = clf_AC_all.model_
classes = encoder.classes_

# 2. Create a "Feature Extractor" model
# We tell it to use the exact same input as your main model, 
# but we set the outputs to be the activations of the first Conv1D layer.
layer_name = trained_keras_model.layers[1].name # Gets the name of the first Conv1D
extractor = tf.keras.models.Model(inputs=trained_keras_model.inputs, 
                                  outputs=trained_keras_model.layers[1].output)

# 3. Pick a single Amplification Curve from your test set to visualize
# We use [0:1] instead of [0] to keep the batch dimension: shape becomes (1, timesteps, 1)
kfolds = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)
splits = kfolds.split(X_AC, ytrue)

for (a, test_index) in splits:
    test_index

sample_index = 100 
single_sample = X_AC[sample_index:sample_index+1] 
true_label = ytrue[test_index][sample_index] # Assuming this matches your loop variables

# 4. Generate the feature maps
# This passes the single curve through the first layer
feature_maps = extractor.predict(single_sample)

# The output shape will be (1, new_timesteps, 16)
print(f"Feature map shape: {feature_maps.shape}") 

# 5. Plot the original curve and the 16 resulting feature maps
fig = plt.figure(figsize=(15, 12))
fig.suptitle(f'Feature Maps for Sample {sample_index} (True Class: {classes[true_label]})', 
             fontsize=16, weight='bold')

# Plot Original Signal at the top
ax_orig = plt.subplot2grid((5, 4), (0, 0), colspan=4)
ax_orig.plot(single_sample.flatten(), color='black', linewidth=2)
ax_orig.set_title('Original Raw Amplification Curve')
ax_orig.set_xlim(0, single_sample.shape[1])
ax_orig.grid(alpha=0.3)

# Plot the 16 Feature Maps below it
for i in range(16):
    ax = plt.subplot2grid((5, 4), (i // 4 + 1, i % 4))
    
    # Extract the 1D signal for filter i
    fmap_signal = feature_maps[0, :, i]
    
    # We use a different color to distinguish it from the raw input
    ax.plot(fmap_signal, color='darkorange')
    ax.set_title(f'Filter {i} Activation', fontsize=10)
    ax.set_xlim(0, feature_maps.shape[1])
    ax.grid(alpha=0.3)
    
    # Optional: Fill the area under the curve to make strong activations pop out
    ax.fill_between(range(len(fmap_signal)), fmap_signal, color='orange', alpha=0.3)

plt.tight_layout()
plt.subplots_adjust(top=0.92) # Leave room for the main title
plt.show()

In [ ]:
trained_keras_model = clf_AC_all.model_ 

conv1_layer = trained_keras_model.layers[1] 

# get_weights() returns a list: [weights_array, biases_array]
weights, biases = conv1_layer.get_weights()

print(f"Layer 1 Weights shape: {weights.shape}") # Should be (5, 1, 16)

# 3. Plot the 16 filters in a 4x4 grid
n_filters = weights.shape[2]

fig, axes = plt.subplots(4, 4, figsize=(12, 10))
fig.suptitle('Visualization of 1st Conv1D Layer Kernels (Size: 5)', fontsize=16, weight='bold')

for i, ax in enumerate(axes.flatten()):
    if i < n_filters:
        # Extract the kernel: all 5 timesteps, channel 0, filter i
        kernel_values = weights[:, 0, i] 
        
        # Plot as a line with markers
        ax.plot(kernel_values, marker='o', color='b', linewidth=2)
        ax.set_title(f'Filter {i}')
        ax.set_xticks(range(weights.shape[0]))
        ax.axhline(0, color='black', linewidth=0.5, linestyle='--') # Add a zero line
    
    ax.grid(True, alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def visualize_best_feature_maps(model_wrapper, X, y_true, class_names):
    """
    Finds the most confidently predicted sample for each class and plots 
    the raw (pre-activation) feature maps from the first Conv1D layer.
    """
    # 1. Access the trained Keras model and get weights
    trained_keras_model = model_wrapper.model_
    weights, biases = trained_keras_model.layers[1].get_weights()
    n_filters = weights.shape[2]
    
    # 2. Get prediction probabilities for the entire dataset
    print("Calculating prediction probabilities to find the best samples...")
    y_probs = model_wrapper.predict_proba(X)
    
    # 3. Iterate through each unique class
    unique_classes = np.unique(y_true)
    
    for cls_idx in unique_classes:
        # Find all indices where the true label matches the current class
        true_indices = np.where(y_true == cls_idx)[0]
        
        if len(true_indices) == 0:
            continue
            
        # 4. Find the sample with the highest probability for this specific class
        class_probs = y_probs[true_indices, cls_idx]
        best_local_idx = np.argmax(class_probs)
        best_global_idx = true_indices[best_local_idx]
        best_confidence = class_probs[best_local_idx]
        
        class_name = class_names[cls_idx]
        single_sample_1d = X[best_global_idx].flatten()
        
        # 5. Prepare the main plot for this class
        fig = plt.figure(figsize=(15, 12))
        fig.suptitle(f'Raw Feature Maps | True Class: {class_name}\n'
                     f'Sample Index: {best_global_idx} | Model Confidence: {best_confidence*100:.2f}%', 
                     fontsize=16, weight='bold')

        # Plot Original Signal at the top
        ax_orig = plt.subplot2grid((5, 4), (0, 0), colspan=4)
        ax_orig.plot(single_sample_1d, color='black', linewidth=2)
        ax_orig.set_title('Original Raw Amplification Curve')
        ax_orig.set_xlim(0, len(single_sample_1d))
        ax_orig.grid(alpha=0.3)

        # 6. Manually calculate and plot the 16 raw Feature Maps
        for i in range(n_filters):
            ax = plt.subplot2grid((5, 4), (i // 4 + 1, i % 4))
            
            kernel = weights[:, 0, i]
            bias = biases[i]
            
            # MANUALLY perform 1D Convolution (Cross-Correlation)
            raw_fmap_signal = np.correlate(single_sample_1d, kernel, mode='valid') + bias
            
            # Plot the raw pre-activation signal
            ax.plot(raw_fmap_signal, color='purple', linewidth=1.5)
            
            # Add a red dashed line at zero 
            ax.axhline(0, color='red', linestyle='--', linewidth=1.2, alpha=0.8)
            
            # Fill the positive (kept) and negative (crushed) areas
            ax.fill_between(range(len(raw_fmap_signal)), raw_fmap_signal, 0, 
                            where=(raw_fmap_signal > 0), color='green', alpha=0.2)
            ax.fill_between(range(len(raw_fmap_signal)), raw_fmap_signal, 0, 
                            where=(raw_fmap_signal < 0), color='red', alpha=0.1)
            
            ax.set_title(f'Filter {i} (Raw Math)', fontsize=10)
            ax.set_xlim(0, len(raw_fmap_signal))
            ax.grid(alpha=0.3)

        plt.tight_layout()
        plt.subplots_adjust(top=0.92)
        plt.show()

# ==========================================
# How to call the function:
# ==========================================
# Assuming clf_AC_all is trained, and X_AC, ytrue, and classes are available:

visualize_best_feature_maps(
    model_wrapper=clf_AC_all, 
    X=X_AC, 
    y_true=ytrue, 
    class_names=classes
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def visualize_layer2_feature_maps(model_wrapper, X, y_true, class_names):
    """
    Finds the most confidently predicted sample for each class and plots 
    the raw (pre-activation) feature maps from the SECOND Conv1D layer.
    """
    trained_model = model_wrapper.model_
    
    # 1. Get weights for BOTH layers
    w1, b1 = trained_model.layers[1].get_weights() # Shape: (5, 1, 16)
    w2, b2 = trained_model.layers[2].get_weights() # Shape: (3, 16, 8)
    
    n_l1_filters = w1.shape[2]
    n_l2_filters = w2.shape[2] # This is 8
    
    print("Calculating prediction probabilities...")
    y_probs = model_wrapper.predict_proba(X)
    unique_classes = np.unique(y_true)
    
    for cls_idx in unique_classes:
        true_indices = np.where(y_true == cls_idx)[0]
        if len(true_indices) == 0:
            continue
            
        # 2. Find the best sample
        class_probs = y_probs[true_indices, cls_idx]
        best_local_idx = np.argmax(class_probs)
        best_global_idx = true_indices[best_local_idx]
        best_confidence = class_probs[best_local_idx]
        
        class_name = class_names[cls_idx]
        single_sample_1d = X[best_global_idx].flatten()
        
        # =========================================================
        # 3. MANUALLY COMPUTE LAYER 1 (And apply ReLU)
        # =========================================================
        l1_length = len(single_sample_1d) - w1.shape[0] + 1
        l1_post_relu = np.zeros((l1_length, n_l1_filters))
        
        for f1 in range(n_l1_filters):
            kernel1 = w1[:, 0, f1]
            raw_l1 = np.correlate(single_sample_1d, kernel1, mode='valid') + b1[f1]
            # Must apply ReLU here, because Layer 2 expects activated inputs!
            l1_post_relu[:, f1] = np.maximum(0, raw_l1) 
            
        # =========================================================
        # 4. MANUALLY COMPUTE LAYER 2 (Pre-Activation)
        # =========================================================
        l2_length = l1_length - w2.shape[0] + 1
        
        fig = plt.figure(figsize=(15, 9))
        fig.suptitle(f'Layer 2 Raw Feature Maps | True Class: {class_name}\n'
                     f'Sample Index: {best_global_idx} | Confidence: {best_confidence*100:.2f}%', 
                     fontsize=16, weight='bold')

        # Plot Original Signal at the top for reference
        ax_orig = plt.subplot2grid((3, 4), (0, 0), colspan=4)
        ax_orig.plot(single_sample_1d, color='black', linewidth=2)
        ax_orig.set_title('Original Raw Amplification Curve')
        ax_orig.set_xlim(0, len(single_sample_1d))
        ax_orig.grid(alpha=0.3)

        # Plot the 8 Layer 2 Feature Maps
        for f2 in range(n_l2_filters):
            ax = plt.subplot2grid((3, 4), (f2 // 4 + 1, f2 % 4))
            
            raw_l2_fmap = np.zeros(l2_length)
            
            # The Multi-Channel Convolution: Summing across all 16 Layer 1 outputs
            for c in range(n_l1_filters):
                kernel2_channel = w2[:, c, f2]
                raw_l2_fmap += np.correlate(l1_post_relu[:, c], kernel2_channel, mode='valid')
            
            # Add Layer 2 bias once per filter
            raw_l2_fmap += b2[f2]
            
            # Plotting
            ax.plot(raw_l2_fmap, color='purple', linewidth=1.5)
            ax.axhline(0, color='red', linestyle='--', linewidth=1.2, alpha=0.8)
            ax.fill_between(range(len(raw_l2_fmap)), raw_l2_fmap, 0, 
                            where=(raw_l2_fmap > 0), color='green', alpha=0.2)
            ax.fill_between(range(len(raw_l2_fmap)), raw_l2_fmap, 0, 
                            where=(raw_l2_fmap < 0), color='red', alpha=0.1)
            
            ax.set_title(f'L2 Filter {f2} (Raw Math)', fontsize=10)
            ax.set_xlim(0, len(raw_l2_fmap))
            ax.grid(alpha=0.3)

        plt.tight_layout()
        plt.subplots_adjust(top=0.88)
        plt.show()

# Run the function
visualize_layer2_feature_maps(clf_AC_all, X_AC, ytrue, classes)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def visualize_feature_maps_grid(model_wrapper, X, y_true, class_names):
    """
    Creates a grid visualization where columns represent the best sample for each class,
    Row 0 is the original curve, and Rows 1-N are the raw feature maps for each filter.
    """
    # 1. Access the trained Keras model and get weights
    trained_keras_model = model_wrapper.model_
    weights, biases = trained_keras_model.layers[1].get_weights()
    n_filters = weights.shape[2]
    
    print("Calculating prediction probabilities to find the best samples...")
    y_probs = model_wrapper.predict_proba(X)
    
    unique_classes = np.unique(y_true)
    n_classes = len(unique_classes)
    
    # Dictionary to store the best sample index and confidence for each class
    best_samples = {}
    for cls_idx in unique_classes:
        true_indices = np.where(y_true == cls_idx)[0]
        if len(true_indices) == 0:
            continue
            
        class_probs = y_probs[true_indices, cls_idx]
        best_local_idx = np.argmax(class_probs)
        best_global_idx = true_indices[best_local_idx]
        best_confidence = class_probs[best_local_idx]
        
        best_samples[cls_idx] = (best_global_idx, best_confidence)

    # 2. Setup the grid layout
    nrows = n_filters + 1  # 1 row for original curve + 1 for each filter
    ncols = len(best_samples)
    
    # Make the figure dynamically scale based on classes and filters
    # squeeze=False ensures 'axes' is always a 2D array, even if there's only 1 class
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 1.8 * nrows), squeeze=False)
    fig.suptitle('Raw Feature Maps Comparison Across Classes', fontsize=20, weight='bold', y=0.99)

    # 3. Populate the grid column by column (class by class)
    for col_idx, (cls_idx, (best_global_idx, conf)) in enumerate(best_samples.items()):
        class_name = class_names[cls_idx]
        single_sample_1d = X[best_global_idx].flatten()
        
        # --- ROW 0: Plot the Original Signal ---
        ax_orig = axes[0, col_idx]
        ax_orig.plot(single_sample_1d, color='black', linewidth=2)
        ax_orig.set_title(f'{class_name}\n(Idx: {best_global_idx} | Conf: {conf*100:.1f}%)', 
                          fontsize=14, weight='bold')
        ax_orig.set_xlim(0, len(single_sample_1d))
        ax_orig.grid(alpha=0.3)
        
        # Only label the Y-axis on the leftmost column
        if col_idx == 0:
            ax_orig.set_ylabel('Original\nCurve', fontsize=12, weight='bold')

        # --- ROWS 1 to N: Plot the Feature Maps ---
        for f_idx in range(n_filters):
            row_idx = f_idx + 1
            ax_fmap = axes[row_idx, col_idx]
            
            kernel = weights[:, 0, f_idx]
            bias = biases[f_idx]
            
            # MANUALLY perform 1D Convolution (Cross-Correlation)
            raw_fmap_signal = np.correlate(single_sample_1d, kernel, mode='valid') + bias
            
            ax_fmap.plot(raw_fmap_signal, color='purple', linewidth=1.5)
            
            # Add a red dashed line at zero 
            ax_fmap.axhline(0, color='red', linestyle='--', linewidth=1.2, alpha=0.8)
            
            # Fill the positive (kept) and negative (crushed) areas
            ax_fmap.fill_between(range(len(raw_fmap_signal)), raw_fmap_signal, 0, 
                                 where=(raw_fmap_signal > 0), color='green', alpha=0.2)
            ax_fmap.fill_between(range(len(raw_fmap_signal)), raw_fmap_signal, 0, 
                                 where=(raw_fmap_signal < 0), color='red', alpha=0.1)
            
            ax_fmap.set_xlim(0, len(raw_fmap_signal))
            ax_fmap.grid(alpha=0.3)
            
            # Label Y-axis for the leftmost column to identify the filter number
            if col_idx == 0:
                ax_fmap.set_ylabel(f'Filter {f_idx}', fontsize=12, weight='bold')

    plt.tight_layout()
    # Adjust spacing so the main title doesn't overlap the column titles
    plt.subplots_adjust(top=0.96, hspace=0.3) 
    plt.show()

# ==========================================
# How to call the function:
# ==========================================
# visualize_feature_maps_grid(clf_AC_all, X_AC, ytrue, classes)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def visualize_feature_maps_per_kernel_axes(model_wrapper, X, y_true, class_names):
    """
    Creates a grid visualization with specifically targeted shared axes:
    - Kernel masks share their own X and Y limits.
    - Original curves share their own X and Y limits.
    - Feature maps share the same X axis length.
    - Feature maps share the Y axis PER KERNEL (Row-based sharing).
    """
    # 1. Access the trained Keras model and get weights
    trained_keras_model = model_wrapper.model_
    weights, biases = trained_keras_model.layers[1].get_weights()
    n_filters = weights.shape[2]
    kernel_size = weights.shape[0]
    
    print("Calculating prediction probabilities to find the best samples...")
    y_probs = model_wrapper.predict_proba(X)
    
    unique_classes = np.unique(y_true)
    
    # Dictionary to store the best sample index and confidence for each class
    best_samples = {}
    for cls_idx in unique_classes:
        true_indices = np.where(y_true == cls_idx)[0]
        if len(true_indices) == 0:
            continue
            
        class_probs = y_probs[true_indices, cls_idx]
        best_local_idx = np.argmax(class_probs)
        best_global_idx = true_indices[best_local_idx]
        best_confidence = class_probs[best_local_idx]
        
        best_samples[cls_idx] = (best_global_idx, best_confidence)

    # ==========================================
    # 2. Pre-calculate Limits for Shared Axes
    # ==========================================
    # Limits for Kernels
    k_min, k_max = weights.min(), weights.max()
    k_margin = (k_max - k_min) * 0.1 if k_max != k_min else 0.1
    
    orig_min, orig_max = float('inf'), float('-inf')
    
    # Arrays to track the min and max for EACH filter independently
    fmap_mins = np.full(n_filters, float('inf'))
    fmap_maxs = np.full(n_filters, float('-inf'))
    
    # Run through to find limits
    for cls_idx, (best_global_idx, _) in best_samples.items():
        sample = X[best_global_idx].flatten()
        orig_min = min(orig_min, sample.min())
        orig_max = max(orig_max, sample.max())
        
        for f_idx in range(n_filters):
            kernel = weights[:, 0, f_idx]
            bias = biases[f_idx]
            fmap = np.correlate(sample, kernel, mode='valid') + bias
            
            # Update the specific min/max for this individual filter
            fmap_mins[f_idx] = min(fmap_mins[f_idx], fmap.min())
            fmap_maxs[f_idx] = max(fmap_maxs[f_idx], fmap.max())
            
    o_margin = (orig_max - orig_min) * 0.1 if orig_max != orig_min else 0.1
    
    # Calculate margins for each filter
    f_margins = [(max_val - min_val) * 0.1 if max_val != min_val else 0.1 
                 for min_val, max_val in zip(fmap_mins, fmap_maxs)]

    # Lengths for X-axes
    orig_length = len(X[list(best_samples.values())[0][0]].flatten())
    fmap_length = orig_length - kernel_size + 1

    # ==========================================
    # 3. Setup the Grid Layout
    # ==========================================
    nrows = n_filters + 1 
    ncols = len(best_samples) + 1 
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 1.8 * nrows), squeeze=False)
    fig.suptitle('Kernel Shapes & Feature Maps (Y-Axis Shared Per Row)', fontsize=20, weight='bold', y=0.99)

    # ==========================================
    # COLUMN 0: Plot the Kernel Masks
    # ==========================================
    axes[0, 0].axis('off')
    axes[0, 0].text(0.5, 0.5, 'Kernel\nMasks', ha='center', va='center', fontsize=16, weight='bold')

    for f_idx in range(n_filters):
        row_idx = f_idx + 1
        ax_kernel = axes[row_idx, 0]
        
        kernel = weights[:, 0, f_idx]
        
        ax_kernel.plot(kernel, marker='o', color='blue', linewidth=2, markersize=5)
        ax_kernel.axhline(0, color='black', linestyle='--', linewidth=1)
        ax_kernel.set_title(f'Weights (Bias: {biases[f_idx]:.2f})', fontsize=10)
        ax_kernel.set_ylabel(f'Filter {f_idx}', fontsize=14, weight='bold')
        
        # Shared axes for kernels
        ax_kernel.set_xlim(-0.5, kernel_size - 0.5)
        ax_kernel.set_ylim(k_min - k_margin, k_max + k_margin)
        ax_kernel.set_xticks(range(kernel_size))
        ax_kernel.grid(alpha=0.3)

    # ==========================================
    # COLUMNS 1 to N: Plot the Classes
    # ==========================================
    for c_idx, (cls_idx, (best_global_idx, conf)) in enumerate(best_samples.items()):
        col_idx = c_idx + 1
        
        class_name = class_names[cls_idx]
        single_sample_1d = X[best_global_idx].flatten()
        
        # --- ROW 0: Plot the Original Signal ---
        ax_orig = axes[0, col_idx]
        ax_orig.plot(single_sample_1d, color='black', linewidth=2)
        ax_orig.set_title(f'{class_name}\n(Idx: {best_global_idx} | Conf: {conf*100:.1f}%)', 
                          fontsize=14, weight='bold')
        
        # Shared axes for original curves
        ax_orig.set_xlim(0, orig_length - 1)
        ax_orig.set_ylim(orig_min - o_margin, orig_max + o_margin)
        ax_orig.grid(alpha=0.3)
        
        if col_idx == 1:
             ax_orig.set_ylabel('Original\nCurve', fontsize=12, weight='bold')

        # --- ROWS 1 to N: Plot the Feature Maps ---
        for f_idx in range(n_filters):
            row_idx = f_idx + 1
            ax_fmap = axes[row_idx, col_idx]
            
            kernel = weights[:, 0, f_idx]
            bias = biases[f_idx]
            
            raw_fmap_signal = np.correlate(single_sample_1d, kernel, mode='valid') + bias
            
            ax_fmap.plot(raw_fmap_signal, color='purple', linewidth=1.5)
            ax_fmap.axhline(0, color='red', linestyle='--', linewidth=1.2, alpha=0.8)
            
            ax_fmap.fill_between(range(len(raw_fmap_signal)), raw_fmap_signal, 0, 
                                 where=(raw_fmap_signal > 0), color='green', alpha=0.2)
            ax_fmap.fill_between(range(len(raw_fmap_signal)), raw_fmap_signal, 0, 
                                 where=(raw_fmap_signal < 0), color='red', alpha=0.1)
            
            # Apply Row-Specific (Per-Kernel) shared Y-axis
            ax_fmap.set_xlim(0, fmap_length - 1)
            ax_fmap.set_ylim(fmap_mins[f_idx] - f_margins[f_idx], fmap_maxs[f_idx] + f_margins[f_idx])
            ax_fmap.grid(alpha=0.3)

    plt.tight_layout()
    plt.subplots_adjust(top=0.96, hspace=0.3) 
    plt.show()

visualize_feature_maps_per_kernel_axes(clf_AC_all, X_AC, ytrue, classes)